In [36]:
import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import chess
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, BatchSampler, RandomSampler, SequentialSampler
from tqdm.auto import tqdm
import time

In [37]:
data_dir     = '/work/cssema416/202710students/15'
parquet_path = f'{data_dir}/dataset.parquet'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

N_GAMES     = 100000
SAVE_SPLITS = True

BATCH  = 256
LR     = .0003
EPOCHS = 100
PATIENCE = 3 

ELO_SCALE = 0.3
USE_AMP = True
USE_COMPILE = True
TRAIN_FRACTION = 1.0

torch.set_float32_matmul_precision('high')
print('Using device:', device)

Using device: cuda


In [38]:
def read_games(path, n_games):
    pf = pq.ParquetFile(path)
    games_per_group = pf.metadata.num_rows / pf.num_row_groups
    n_groups = min(pf.num_row_groups, max(50, int(2 * n_games / games_per_group) + 1))
    groups   = np.linspace(0, pf.num_row_groups - 1, n_groups).astype(int)

    df = pf.read_row_groups(groups.tolist(), columns=['white_elo', 'black_elo', 'moves_uci']).to_pandas()
    df = df.dropna(subset=['white_elo', 'black_elo'])
    df = df.sample(n=min(n_games, len(df)), random_state=0)
    print(f'Read {n_groups} row groups, kept {len(df)} games')
    games = [list(moves) for moves in df['moves_uci']]
    elos  = df[['white_elo', 'black_elo']].to_numpy().astype(np.int16)
    return games, elos

In [39]:
def board_to_array(board):
    arr = np.zeros(64, dtype=np.uint8)
    for square, piece in board.piece_map().items():
        arr[square] = piece.piece_type + (0 if piece.color == chess.WHITE else 6)
    return arr

In [40]:
def encode_side(board):
    castling = (board.has_kingside_castling_rights(chess.WHITE) * 1 + board.has_queenside_castling_rights(chess.WHITE) * 2
                + board.has_kingside_castling_rights(chess.BLACK) * 4 + board.has_queenside_castling_rights(chess.BLACK) * 8)
    ep = chess.square_file(board.ep_square) if board.has_legal_en_passant() else 8
    return (0 if board.turn == chess.WHITE else 1) + 2 * castling + 32 * ep


def replay_game(moves):
    board = chess.Board()
    boards, sides, played = [], [], []
    for uci in moves:
        try:
            move = board.parse_uci(uci)
        except ValueError:
            break
        if not board.is_legal(move):
            break
        boards.append(board_to_array(board))
        sides.append(encode_side(board))
        played.append(move.uci())
        board.push(move)
    return np.array(boards, dtype=np.uint8).reshape(-1, 64), sides, played

In [41]:
games, elos = read_games(parquet_path, N_GAMES)
positions = [replay_game(moves) for moves in tqdm(games)]
print(f'Total positions: {sum(len(played) for _, _, played in positions)}')

Read 201 row groups, kept 100000 games


  1%|▏         | 1481/100000 [00:05<05:36, 292.55it/s]

KeyboardInterrupt: 

  1%|▏         | 1481/100000 [00:19<05:36, 292.55it/s]

In [ ]:
vocab = {move: i for i, move in enumerate(sorted({m for _, _, played in positions for m in played}))}
with open(f'{data_dir}/vocab.json', 'w') as f:
    json.dump(vocab, f)

NUM_MOVES = len(vocab)
print('NUM_MOVES:', NUM_MOVES)

NUM_MOVES: 1913


In [ ]:
def legal_masks(played):
    board = chess.Board()
    masks = np.zeros((len(played), NUM_MOVES), dtype=bool)
    for row, uci in zip(masks, played):
        for move in board.legal_moves:
            idx = vocab.get(move.uci())
            if idx is not None:
                row[idx] = True
        board.push_uci(uci)
    return np.packbits(masks, axis=1)

packed_legal = [legal_masks(played) for _, _, played in tqdm(positions)]


100%|██████████| 100000/100000 [11:14<00:00, 148.24it/s]


In [ ]:
buckets    = pd.util.hash_array(np.array([' '.join(moves) for moves in games], dtype=object)) % 100
game_split = np.where(buckets < 80, 'train', np.where(buckets < 90, 'val', 'test'))
print(pd.Series(game_split).value_counts().to_dict())

{'train': 80045, 'val': 10007, 'test': 9948}


In [ ]:
def build_split(name):
    idx     = np.flatnonzero(game_split == name)
    boards  = np.array([b for i in idx for b in positions[i][0]], dtype=np.uint8).reshape(-1, 64)
    side    = np.array([s for i in idx for s in positions[i][1]], dtype=np.int64)
    elo     = np.array([elos[i] for i in idx for _ in positions[i][2]], dtype=np.int16).reshape(-1, 2)
    targets = np.array([vocab[m] for i in idx for m in positions[i][2]], dtype=np.int64)
    legal   = np.concatenate([packed_legal[i] for i in idx]).reshape(-1, (NUM_MOVES + 7) // 8)
    return boards, side, elo, targets, legal

In [ ]:
data = {name: build_split(name) for name in ['train', 'val', 'test']}

if SAVE_SPLITS:
    for name, (boards, side, elo, targets, legal) in data.items():
        np.save(f'{data_dir}/boards_{name}.npy',  boards)
        np.save(f'{data_dir}/side_{name}.npy',    side)
        np.save(f'{data_dir}/elo_{name}.npy',     elo)
        np.save(f'{data_dir}/targets_{name}.npy', targets)
        np.save(f'{data_dir}/legal_{name}.npy',   legal)

In [ ]:
def load_split(name):
    boards  = np.load(f'{data_dir}/boards_{name}.npy')
    side    = np.load(f'{data_dir}/side_{name}.npy').astype(np.int64)
    elo     = np.load(f'{data_dir}/elo_{name}.npy').astype(np.int64)
    targets = np.load(f'{data_dir}/targets_{name}.npy').astype(np.int64)
    legal   = np.load(f'{data_dir}/legal_{name}.npy')
    return boards, side, elo, targets, legal

In [ ]:
if SAVE_SPLITS:
    data = {name: load_split(name) for name in ['train', 'val', 'test']}

train_boards, train_side, train_elo, train_targets, train_legal = data['train']
val_boards,   val_side,   val_elo,   val_targets,   val_legal   = data['val']

print(f'Training set size:   {train_boards.shape[0]} samples')
print(f'Validation set size: {val_boards.shape[0]} samples')

Training set size:   6805217 samples
Validation set size: 850145 samples


In [ ]:
class ChessDataset(Dataset):
    def __init__(self, boards, side, elo, targets, legal, num_moves):
        self.boards    = boards
        self.side      = side
        self.elo       = elo
        self.targets   = targets
        self.legal     = legal
        self.num_moves = num_moves

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        return (torch.from_numpy(self.boards[index].astype(np.int64)), torch.from_numpy(self.side[index]),
                torch.from_numpy(self.elo[index]), torch.from_numpy(self.legal[index]),
                torch.from_numpy(self.targets[index]))


def make_loader(dataset, batch_size, shuffle, fraction=1.0, drop_last=False):
    if shuffle:
        sampler = RandomSampler(dataset, num_samples=int(len(dataset) * fraction))
    else:
        sampler = SequentialSampler(dataset)
    return DataLoader(dataset, sampler=BatchSampler(sampler, batch_size, drop_last=drop_last), batch_size=None)


BIT_SHIFTS = torch.arange(7, -1, -1)

def unpack_legal(packed, num_moves):
    bits = (packed[:, :, None] >> BIT_SHIFTS.to(packed.device)) & 1
    return bits.flatten(1)[:, :num_moves].bool()


In [ ]:
train_dataset = ChessDataset(train_boards, train_side, train_elo, train_targets, train_legal, NUM_MOVES)
val_dataset   = ChessDataset(val_boards,   val_side,   val_elo,   val_targets,   val_legal,   NUM_MOVES)

train_loader = make_loader(train_dataset, BATCH, shuffle=True, fraction=TRAIN_FRACTION, drop_last=True)
val_loader   = make_loader(val_dataset,   BATCH, shuffle=False)

In [ ]:
class PieceGeometricBias(nn.Module):
    def __init__(self, num_heads, num_extra):
        super().__init__()
        sq = torch.arange(64)
        file, rank = sq % 8, sq // 8
        rel = (file[None, :] - file[:, None] + 7) * 15 + (rank[None, :] - rank[:, None] + 7)
        self.register_buffer('rel', rel)
        self.table = nn.Embedding(13 * 15 * 15, num_heads)
        nn.init.zeros_(self.table.weight)
        self.num_extra = num_extra

    def forward(self, boards):
        index = boards[:, :, None] * (15 * 15) + self.rel[None]
        bias = self.table(index).permute(0, 3, 1, 2)
        return F.pad(bias, (0, self.num_extra, 0, self.num_extra))

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super().__init__()
        self.num_heads = num_heads
        self.dropout   = dropout
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.qkv   = nn.Linear(d_model, 3 * d_model)
        self.out   = nn.Linear(d_model, d_model)
        self.ff    = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, bias):
        B, L, D = x.shape
        qkv = self.qkv(self.norm1(x)).view(B, L, 3, self.num_heads, D // self.num_heads)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)
        attn = F.scaled_dot_product_attention(q, k, v, attn_mask=bias.to(q.dtype),
                                              dropout_p=self.dropout if self.training else 0.0)
        x = x + self.drop(self.out(attn.transpose(1, 2).reshape(B, L, D)))
        x = x + self.drop(self.ff(self.norm2(x)))
        return x

In [ ]:
ELO_BUCKET      = 100
NUM_ELO_BUCKETS = 36

class ChessModel(nn.Module):
    def __init__(self, num_moves=1968, d_model=256, num_heads=8, num_layers=6, d_ff=1024, dropout=0.1, elo_scale=1.0):
        super().__init__()
        self.elo_scale  = elo_scale
        self.piece_emb  = nn.Embedding(13, d_model)
        self.square_pos = nn.Parameter(torch.randn(64, d_model) * 0.02)
        self.side_emb   = nn.Embedding(2, d_model)
        self.white_elo  = nn.Embedding(NUM_ELO_BUCKETS, d_model)
        self.black_elo  = nn.Embedding(NUM_ELO_BUCKETS, d_model)
        self.cls        = nn.Parameter(torch.zeros(1, d_model))
        self.castle_emb = nn.Embedding(16, d_model)
        self.ep_emb     = nn.Embedding(9, d_model)
        self.bias   = PieceGeometricBias(num_heads, 6)
        self.blocks = nn.ModuleList([EncoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.norm   = nn.LayerNorm(d_model)
        self.head   = nn.Linear(d_model, num_moves)

    def forward(self, boards, side, elo, legal):
        B = boards.size(0)
        side, castle, ep = side % 2, (side // 2) % 16, side // 32
        bucket = (elo // ELO_BUCKET).clamp(0, NUM_ELO_BUCKETS - 1)
        white, black = self.white_elo(bucket[:, 0]), self.black_elo(bucket[:, 1])
        elo_vec = self.elo_scale * (white + black)
        x = self.piece_emb(boards) + self.square_pos + elo_vec[:, None]
        extra = torch.stack([self.cls.expand(B, -1), self.side_emb(side),
                             self.elo_scale * white, self.elo_scale * black,
                             self.castle_emb(castle), self.ep_emb(ep)], dim=1)
        x = torch.cat([x, extra], dim=1)

        bias = self.bias(boards)
        for block in self.blocks:
            x = block(x, bias)

        logits = self.head(self.norm(x[:, 64]))
        return logits.masked_fill(~legal, float('-inf'))

In [ ]:
GRAD_CLIP = 1.0

def train_epoch(model, loader, optimizer, device, scheduler=None):
    model.train()
    total_loss = torch.zeros((), device=device)
    seen = 0
    for boards, side, elo, legal, targets in loader:
        boards, side, elo, legal, targets = (t.to(device, non_blocking=True) for t in (boards, side, elo, legal, targets))
        legal = unpack_legal(legal, NUM_MOVES)
        with torch.autocast(device_type=device, dtype=torch.bfloat16, enabled=USE_AMP and device == 'cuda'):
            logits = model(boards, side, elo, legal)
        loss = F.cross_entropy(logits.float(), targets)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.detach() * len(targets)
        seen += len(targets)
    return total_loss.item() / seen

In [ ]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct = {k: torch.zeros((), device=device) for k in (1, 3, 5)}
    total_loss = torch.zeros((), device=device)
    for boards, side, elo, legal, targets in loader:
        boards, side, elo, legal, targets = (t.to(device, non_blocking=True) for t in (boards, side, elo, legal, targets))
        legal = unpack_legal(legal, NUM_MOVES)
        with torch.autocast(device_type=device, dtype=torch.bfloat16, enabled=USE_AMP and device == 'cuda'):
            logits = model(boards, side, elo, legal)
        logits = logits.float()
        total_loss += F.cross_entropy(logits, targets, reduction='sum')
        top5 = logits.topk(5, dim=-1).indices
        for k in correct:
            correct[k] += (top5[:, :k] == targets[:, None]).any(dim=1).sum()
    n = len(loader.dataset)
    return total_loss.item() / n, {k: v.item() / n for k, v in correct.items()}

In [ ]:
sanity_loader    = make_loader(ChessDataset(train_boards[:256], train_side[:256], train_elo[:256], train_targets[:256], train_legal[:256], NUM_MOVES), 32, shuffle=True)
sanity_model     = ChessModel(num_moves=NUM_MOVES, elo_scale=ELO_SCALE).to(device)
sanity_optimizer = torch.optim.AdamW(sanity_model.parameters(), lr=LR, weight_decay=0.01)

for epoch in range(30):
    sanity_loss = train_epoch(sanity_model, sanity_loader, sanity_optimizer, device)
    if (epoch + 1) % 10 == 0:
        print(f'Sanity epoch {epoch+1}/30: train loss {sanity_loss:.3f}')
print(f'ln(NUM_MOVES) = {np.log(NUM_MOVES):.3f}')

Sanity epoch 10/30: train loss 1.403
Sanity epoch 20/30: train loss 1.027


KeyboardInterrupt: 

In [ ]:
model = ChessModel(num_moves=NUM_MOVES, elo_scale=ELO_SCALE).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01, fused=device == 'cuda')
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=1)

last_path = f'{data_dir}/chess_model_last.pt'
start_epoch, best_top1, bad_epochs = 0, -1.0, 0
try:
    ck = torch.load(last_path, map_location=device, weights_only=False)
    model.load_state_dict(ck['model'])
    optimizer.load_state_dict(ck['optimizer'])
    scheduler.load_state_dict(ck['scheduler'])
    start_epoch, best_top1, bad_epochs = ck['epoch'] + 1, ck['best_top1'], ck['bad_epochs']
    print(f'Resuming at epoch {start_epoch + 1}')
except FileNotFoundError:
    print('No checkpoint found, starting fresh')

train_model = torch.compile(model) if USE_COMPILE else model
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

NameError: name 'ChessModel' is not defined

In [ ]:
for epoch in range(start_epoch, EPOCHS):
    t0 = time.time()
    train_loss = train_epoch(train_model, train_loader, optimizer, device)   # no scheduler here any more
    t_train = time.time() - t0
    val_loss, val_acc = evaluate(train_model, val_loader, device)
    t_val = time.time() - t0 - t_train
    scheduler.step(val_acc[1])                                               # halves LR if top-1 stalls
    print(f'Epoch {epoch+1} [{t_train:.0f}s train, {t_val:.0f}s val]: train loss {train_loss:.3f}, val loss {val_loss:.3f}, '
          f'top-1 {val_acc[1]:.3f}, top-3 {val_acc[3]:.3f}, top-5 {val_acc[5]:.3f}, lr {optimizer.param_groups[0]["lr"]:.1e}', flush=True)

    if val_acc[1] > best_top1 + 0.001:          # counts as improvement only if top-1 rises by more than 0.1%
        best_top1, bad_epochs = val_acc[1], 0
        torch.save(model.state_dict(), f'{data_dir}/chess_model_best.pt')
    else:
        bad_epochs += 1

    torch.save(dict(model=model.state_dict(), optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(),
                    epoch=epoch, best_top1=best_top1, bad_epochs=bad_epochs), last_path)

    if bad_epochs >= PATIENCE:
        print(f'Converged: no improvement for {PATIENCE} epochs. Best val top-1 {best_top1:.3f}')
        break

/work/cssema416/202710students/15/.venv/lib/python3.11/site-packages/torch/_inductor/compile_fx.py:3634: UserWarning: Quadro RTX 6000 does not support bfloat16 compilation natively, skipping
  warnings.warn(
/work/cssema416/202710students/15/.venv/lib/python3.11/site-packages/triton/knobs.py:504: UserWarning: TRITON_DEFAULT_FP_FUSION is unset: defaulting to FP fusion ENABLED for backward compatibility in Triton 3.9. The default will change to DISABLED in Triton 3.10; set TRITON_DEFAULT_FP_FUSION=1 to keep fusion enabled or =0 to adopt the new default early and silence this warning.
  warn_once(FP_FUSION_DEFAULT_WARNING)
/work/cssema416/202710students/15/.venv/lib/python3.11/site-packages/torch/_inductor/compile_fx.py:3634: UserWarning: Quadro RTX 6000 does not support bfloat16 compilation natively, skipping
  warnings.warn(


Epoch 15 [3045s train, 122s val]: train loss 1.834, val loss 1.917, top-1 0.410, top-3 0.679, top-5 0.791, lr 3.0e-04
Epoch 16 [3022s train, 122s val]: train loss 1.823, val loss 1.914, top-1 0.411, top-3 0.681, top-5 0.792, lr 3.0e-04
Epoch 17 [3027s train, 122s val]: train loss 1.814, val loss 1.910, top-1 0.413, top-3 0.683, top-5 0.793, lr 3.0e-04


In [ ]:
model = ChessModel(num_moves=NUM_MOVES, elo_scale=ELO_SCALE).to(device)
model.load_state_dict(torch.load(f'{data_dir}/chess_model_best.pt', map_location=device))
test_boards, test_side, test_elo, test_targets, test_legal = data['test']
test_loader = make_loader(ChessDataset(test_boards, test_side, test_elo, test_targets, test_legal, NUM_MOVES), BATCH, shuffle=False)

test_loss, test_acc = evaluate(model, test_loader, device)
print(f'Test loss {test_loss:.3f}, top-1 {test_acc[1]:.3f}, top-3 {test_acc[3]:.3f}, top-5 {test_acc[5]:.3f}')

Test loss 1.898, top-1 0.416, top-3 0.685, top-5 0.796
